# Importación de Datos de SteamDB a InfluxDB (Últimos 29 Días)

Este notebook importa **exclusivamente los últimos 29 días** del archivo `steamdb_chart_730.csv` a InfluxDB (`bucketsomorrostro`).

### ¿Por qué solo los últimos 29 días?
En las exportaciones de SteamDB, los datos con más de 1 mes de antigüedad se comprimen en una única media diaria (1 punto por día). A partir de los últimos 29 días, SteamDB conserva la **resolución de alta frecuencia (cada 10 minutos)**, que es la adecuada para que el modelo y Grafana aprendan los patrones horarios reales sin distorsiones.

In [ ]:
# Descomenta si necesitas instalar la librería cliente de InfluxDB:
# !pip install influxdb-client

import os
import csv
import time
from datetime import datetime, timezone, timedelta
from influxdb_client import InfluxDBClient
from influxdb_client.client.write_api import SYNCHRONOUS

## 1. Configuración de InfluxDB y Parámetros

In [ ]:
INFLUX_URL = "http://localhost:8086"
INFLUX_TOKEN = "tokensomorrostro123456789"
INFLUX_ORG = "orgsomorrostro"
INFLUX_BUCKET = "bucketsomorrostro"
MEASUREMENT = "cs2_live_players"
FIELD_NAME = "jugadores"

DIAS_HISTORICO = 29  # Filtrar estrictamente los últimos 29 días de alta resolución

# Ruta al archivo CSV
CSV_PATH = "steamdb_chart_730.csv" if os.path.exists("steamdb_chart_730.csv") else r"D:\Reto0\steamdb_chart_730.csv"
print(f"Usando archivo: {CSV_PATH}")

## 2. Lectura y Filtrado (Solo Últimos 29 Días)
- Se determina la fecha más reciente del CSV y se calcula la fecha de corte (`fecha_max - 29 días`).
- Se descartan todos los datos anteriores a esa fecha (evitando los promedios diarios antiguos).
- Se formatea a Line Protocol nativo de InfluxDB.

In [ ]:
# Paso 1: Encontrar la fecha más reciente del archivo
todas_filas = []
fecha_maxima = None

with open(CSV_PATH, mode="r", encoding="utf-8-sig") as f:
    reader = csv.reader(f, delimiter=";")
    cabeceras = next(reader)
    
    for fila in reader:
        if len(fila) < 2 or not fila[1].strip():
            continue
        fecha_str = fila[0].strip().replace('"', '')
        players_str = fila[1].strip()
        try:
            dt = datetime.strptime(fecha_str, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
            jugadores = float(players_str)
            todas_filas.append((dt, jugadores))
            if fecha_maxima is None or dt > fecha_maxima:
                fecha_maxima = dt
        except Exception:
            continue

fecha_corte = fecha_maxima - timedelta(days=DIAS_HISTORICO)
print(f"Fecha más reciente en CSV: {fecha_maxima}")
print(f"Fecha de corte (29 días atrás): {fecha_corte}")

# Paso 2: Filtrar filas que estén dentro de los últimos 29 días
lineas = []
for dt, jugadores in todas_filas:
    if dt >= fecha_corte:
        ts_ns = int(dt.timestamp()) * 1_000_000_000
        lineas.append(f"{MEASUREMENT} {FIELD_NAME}={jugadores} {ts_ns}")

print(f"Total de puntos filtrados para importar: {len(lineas)} (descartados {len(todas_filas) - len(lineas)} puntos antiguos de media diaria)")
if lineas:
    print(f"Primer punto a importar: {lineas[0]}")
    print(f"Último punto a importar: {lineas[-1]}")

## 3. Envío por Lotes a InfluxDB

In [ ]:
print(f"Conectando a {INFLUX_URL}...")
client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG, timeout=60_000)
write_api = client.write_api(write_options=SYNCHRONOUS)

BATCH = 200
total = len(lineas)
t_inicio = time.time()

for i in range(0, total, BATCH):
    lote = lineas[i:i + BATCH]
    for intento in range(3):
        try:
            write_api.write(bucket=INFLUX_BUCKET, record=lote)
            break
        except Exception as e:
            if intento < 2:
                time.sleep(1)
            else:
                print(f"Error en lote {i}: {e}")
    if (i // BATCH) % 2 == 0 or (i + BATCH >= total):
        print(f"Progreso: {min(i + BATCH, total)}/{total} puntos importados...")

duracion = time.time() - t_inicio
print(f"¡Carga de los últimos 29 días completada exitosamente en {duracion:.2f} segundos!")

## 4. Comprobación y Verificación en InfluxDB

In [ ]:
query_api = client.query_api()
query = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: 0)
  |> filter(fn: (r) => r["_measurement"] == "{MEASUREMENT}")
  |> filter(fn: (r) => r["_field"] == "{FIELD_NAME}")
  |> count()
'''

tablas = query_api.query(query)
for t in tablas:
    for r in t.records:
        print(f"Total de puntos actuales en '{MEASUREMENT}.{FIELD_NAME}': {r.get_value()}")

client.close()